In [0]:
class bronze:
    def __init__(self):
        self.data_path='/Volumes/cat_spark_streaming/bronze'
        self.catalog_name="cat_spark_streaming"
        self.schema_name="bronze"
        self.table_name="bronze_invoice"
    
    def get_schema(self):
        from pyspark.sql.types import StringType,FloatType,ArrayType,StructType,IntegerType,StructField
        json_schema= StructType([
                    StructField("InvoiceNumber", StringType()),
                    StructField('CreatedTime',StringType()),
                    StructField('StoreID', StringType()),
                    StructField('PosID',StringType()),
                    StructField('CashierID',StringType()),
                    StructField('CustomerType', StringType()),
                    StructField('CustomerCardNo', StringType()),
                    StructField('TotalAmount', FloatType()),
                    StructField('NumberOfItems', IntegerType()),
                    StructField('PaymentMethod', StringType()),
                    StructField('TaxableAmount', FloatType()),
                    StructField('CGST', FloatType()),
                    StructField('SGST', FloatType()),
                    StructField('CESS',  FloatType()),
                    StructField('DeliveryType', StringType()),
                    ##StructField('inputFileName', StringType()),
                    StructField('DeliveryAddress', StructType([
                        StructField('AddressLine', StringType()),
                        StructField('City', StringType()),
                        StructField('State', StringType()),
                        StructField('PinCode', StringType()),
                        StructField('ContactNumber', StringType())
                    ])),
                    StructField('InvoiceLineItems', ArrayType(
                        StructType([
                            StructField('ItemCode', StringType()),
                            StructField('ItemDescription', StringType()),
                            StructField('ItemPrice', FloatType()),
                            StructField('ItemQty',IntegerType()),
                            StructField('TotalValue',FloatType())
                        ])
                    ))
                ])
        
        return json_schema
    

    def read_raw_data(self):
        from pyspark.sql.functions import col
        return(
                spark.readStream
                    .format('json')
                    .schema(self.get_schema())
                    .option('cleanSource', 'archive')
                    .option('sourceArchiveDir', f'{self.data_path}/archive')
                    .load(f'{self.data_path}/data/invoice-data-files') 
                    .withColumn('inputFileName',col("_metadata.file_name"))
        )  


    def write_to_table(self,raw_df):
        query=(
                raw_df.writeStream
                    .format('delta')
                    .queryName('bronze-ingestion')
                    .outputMode('append')
                    .option('checkpointLocation',f'{self.data_path}/data/checkpoint/{self.table_name}')
                    .trigger(availableNow=True)
                    .toTable(f'{self.catalog_name}.{self.schema_name}.{self.table_name}')

        )
        return query
    
    def process(self):
        print(f'Starting Bronze ingestion...',end='')
        raw_df=self.read_raw_data()
        query=self.write_to_table(raw_df)
        print(f'complete')
        return query

In [0]:
obj=bronze()
obj.process()

In [0]:
%sql
select * from cat_spark_streaming.bronze.bronze_invoice;

In [0]:
class silver:
    def __init__(self):
        self.data_path='/Volumes/cat_spark_streaming/silver'
        self.catalog_name="cat_spark_streaming"
        self.schema_name="silver"
        self.table_name="silver_invoice"
 
    def read_raw_data(self):
        return (
            spark.readStream
                .table(f'cat_spark_streaming.bronze.bronze_invoice')
        )

    def json_transformation(self,df):
        from pyspark.sql.functions import explode_outer,col
        return (
                df.select(
                    'InvoiceNumber',
                    'CreatedTime',
                    'StoreID',
                    'PosID',
                    'CashierID',
                    'CustomerType',
                    'CustomerCardNo',
                    'TotalAmount',
                    'NumberOfItems',
                    'PaymentMethod',
                    'TaxableAmount',
                    'CGST',
                    'SGST',
                    'CESS',
                    'DeliveryType',
                    'DeliveryAddress.AddressLine',
                    'DeliveryAddress.City',
                    'DeliveryAddress.State',
                    'DeliveryAddress.PinCode',
                    'DeliveryAddress.ContactNumber',
                    explode_outer(col('InvoiceLineItems')).alias('InvoiceLineItems'))
                        .select( '*',
                                'InvoiceLineItems.ItemCode',
                                'InvoiceLineItems.ItemDescription',
                                'InvoiceLineItems.ItemPrice',
                                'InvoiceLineItems.ItemQty',
                                'InvoiceLineItems.TotalValue'
            ).drop('InvoiceLineItems')
        )

    def write_to_table(self,raw_df):
        query=(
                raw_df.writeStream
                    .format('delta')
                    .queryName('silver-ingestion')
                    .outputMode('append')
                    .option('checkpointLocation',f'{self.data_path}/checkpoint/{self.table_name}')
                    .option('maxFilesPerTrigger',2)
                    .trigger(availableNow=True)
                    .toTable(f'{self.catalog_name}.{self.schema_name}.{self.table_name}')
        )
        return query

    def process(self):
        print(f'Starting Silver ingestion...',end='')
        raw_df=self.read_raw_data()
        transformed_df=self.json_transformation(raw_df)
        query=self.write_to_table(transformed_df)
        print(f'complete')
        return query

In [0]:
obj=silver()
obj.process()

In [0]:
class medallionTestSuite:
    def __init__(self):
        self.base_path='/Volumes/cat_spark_streaming/landing/datasets/json_files/'
        self.test_path='/Volumes/cat_spark_streaming/bronze/data/invoice-data-files/'
        self.archive_path='/Volumes/cat_spark_streaming/bronze/archive/'
        self.catalog_name="cat_spark_streaming"
        self.bronze_schema="bronze"        
        self.silver_schema="silver"
        self.bronze_table="bronze_invoice"
        self.silver_table="silver_invoice"

    def clean(self):
        print('Cleaning')
        dbutils.fs.rm(f'{self.test_path}',True)
        dbutils.fs.rm(f'{self.archive_path}',True)
        dbutils.fs.rm('/Volumes/cat_spark_streaming/bronze/data/checkpoint/',True)
        dbutils.fs.rm('/Volumes/cat_spark_streaming/silver/checkpoint/',True)
        print('Cleaned and Recreating directories: ')
        dbutils.fs.mkdirs(self.test_path)
        dbutils.fs.mkdirs(self.archive_path)
        dbutils.fs.mkdirs('/Volumes/cat_spark_streaming/bronze/data/checkpoint/')
        dbutils.fs.mkdirs('/Volumes/cat_spark_streaming/silver/checkpoint/')        
        spark.sql(f"drop table if exists {self.catalog_name}.{self.bronze_schema}.{self.bronze_table}")
        spark.sql(f"drop table if exists {self.catalog_name}.{self.silver_schema}.{self.silver_table}")


if __name__=="__main__":
    obj=medallionTestSuite()
    obj.clean()